# BTP: supervised ECG experiments

Primary entry point for **Self-Supervised Learning for ECG Classification Using Mamba**.
This stage establishes the supervised baseline only. Original notebooks are preserved.

**Dataset correction:** official PTB-XL v1.0.3 has **23 diagnostic subclasses**, not 24.
The head dimension is derived from the complete official mapping, with no frequency cutoff.
Superclasses use `NORM, MI, STTC, CD, HYP`; subclasses use sorted official names.

Read `README.md` for installation, dataset version verification and CUDA wheel setup.
The cells below do not start training until you explicitly enable it.

## 1. Locate the repository

On Kaggle, attach PTB-XL **v1.0.3** and the repository source as datasets. Set `REPO_DIR`
to the folder containing `src/` and `DATA_DIR` to the folder containing both metadata CSVs.
No need to attach or execute the three original notebooks.

In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path(os.environ.get("BTP_REPO_DIR", ".")).resolve()
DATA_DIR = os.environ.get("PTBXL_DATA_DIR", "")
assert (REPO_DIR / "src").is_dir(), "Set BTP_REPO_DIR to the repository root"
sys.path.insert(0, str(REPO_DIR))

from src.config import DataConfig, ModelConfig, ExperimentConfig
from src.data import load_ptbxl
from src.experiments import run_experiment
from src.utils import environment_info

environment_info()

## 2. Central configuration

Reference training source: 384 embedding/hidden width, 8 bidirectional blocks,
16 state size, convolution width 4, expansion 2, dropout 0.45, mean-attention pooling.
Source optimizer settings: AdamW 3e-4, weight decay 5e-4, warmup 3, smoothing 0.05,
positive weights capped at 8, clipping 1, early stopping patience 15.
Source budget was 50 epochs; this experiment defaults to 20. Reduce batch size on OOM.

CPU tests: run `python -m pytest -q` from the repository. They do not install Mamba.

In [ ]:
CONFIG = ExperimentConfig(
    epochs=20,
    batch_size=64,
    data=DataConfig(data_dir=DATA_DIR, num_workers=2, augment=True),
    model=ModelConfig(),
    output_root="/kaggle/working/btp_outputs" if Path("/kaggle/working").exists() else "outputs",
)
RUN_TRAINING = False  # Set True only after installing compatible CUDA packages and attaching data.

## 3. Optional data audit

Loading fits streaming normalization on **processed training-fold** records only; it
reads every training waveform once. Both tasks fit from the same records. Inspect the
per-split statistics; patient overlap, missing/corrupt waveforms or invalid mappings fail clearly.
Records without diagnostic labels are counted and excluded; rare classes remain as columns.

In [ ]:
RUN_DATA_AUDIT = False
if RUN_DATA_AUDIT:
    data = load_ptbxl(task="subclass", config=CONFIG.data)
    print(data.class_names)
    print("Excluded unlabeled records:", len(data.dropped_ids))
    display(data.statistics)
    loaders = data.loaders(CONFIG.batch_size)
    x, y, ids = next(iter(loaders["val"]))
    print(x.shape, y.shape, ids.shape)

## 4. Superclass experiment (5 labels)

Each run uses a unique directory. The best validation macro AUROC selects the checkpoint.
Validation alone selects per-class F1 thresholds; test fold 10 is evaluated with frozen thresholds.
Training uses binary multi-label BCE logits and sigmoid probabilities, never softmax.

In [ ]:
if RUN_TRAINING:
    superclass_result = run_experiment(
        model_name="mamba", task="superclass", epochs=20, config=CONFIG
    )
    display(superclass_result["history"])
    display(superclass_result["summary"])

## 5. Subclass experiment (all 23 official labels)

This uses the same signals, folds, preprocessing and metric definitions as the superclass task.

In [ ]:
if RUN_TRAINING:
    subclass_result = run_experiment(
        model_name="mamba", task="subclass", epochs=20, config=CONFIG
    )
    display(subclass_result["summary"])

## 6. Checkpoints, exports and future adapters

Each run writes `config.json`, `environment.json`, `data_manifest.json`, class frequencies,
`best.pth`, `last.pth`, history CSV, validation thresholds, predictions, overall/per-class metrics,
confusion matrix CSVs/figures and loss curves. Missing ranking metrics are JSON `null`.

To resume an interrupted run use the **same configuration and original output directory**:
```python
result = run_experiment(config=CONFIG, task="superclass",
    run_dir="/kaggle/working/btp_outputs/<original_run>",
    resume_from="/kaggle/working/btp_outputs/<original_run>/last.pth")
```
To load for inference, instantiate the model with the saved config, then call
`src.training.load_checkpoint(path, model, class_names=ordered_names)`. The checkpoint
contains fitted normalization, fold IDs, vocabulary and metadata hashes.
For exact CPU resume use `num_workers=0`; multi-worker augmentation restart is not bitwise exact.
Mamba fused GPU kernels may be nondeterministic even with fixed seeds.

Future adapters register a factory with `register_model(name, factory)` and return raw
`(B,C)` logits for `(B,12,T)` signals. No extra architectures are included now.
`MambaEncoder` and `forward_features` expose the head-free representation for future SSL;
this notebook performs no self-supervised pretraining.